In [ ]:
import sys
import subprocess

required = ["sentence-transformers", "datasets", "scipy", "pandas", "transformers"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])


In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 128 if device == "mps" else 64
max_length = 128
target_subset_size = 300
num_bins = 6
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "max_length": max_length,
    "target_subset_size": target_subset_size,
    "num_bins": num_bins,
    "seed": seed,
})


In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df = df.dropna(subset=["sentence1", "sentence2", "label"]).reset_index(drop=True)

bin_edges = np.linspace(0.0, 5.0, num_bins + 1)
bin_labels = [f"[{bin_edges[i]:.2f},{bin_edges[i+1]:.2f}{')' if i < num_bins - 1 else ']'}" for i in range(num_bins)]
df["label_bin_index"] = pd.cut(
    df["label"],
    bins=bin_edges,
    labels=False,
    include_lowest=True,
    right=True,
).astype(int)
df["label_bin"] = df["label_bin_index"].map({i: bin_labels[i] for i in range(num_bins)})

base_per_bin = target_subset_size // num_bins
remainder = target_subset_size % num_bins
desired_counts = {i: base_per_bin + (1 if i < remainder else 0) for i in range(num_bins)}

sampled_parts = []
for i in range(num_bins):
    part = df[df["label_bin_index"] == i].sample(n=desired_counts[i], random_state=seed, replace=False)
    sampled_parts.append(part)

subset_df = pd.concat(sampled_parts, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)

bin_counts = subset_df["label_bin"].value_counts().sort_index()
print({
    "original_num_examples": len(df),
    "subset_num_examples": len(subset_df),
    "desired_counts": desired_counts,
    "actual_bin_counts": bin_counts.to_dict(),
    "label_mean": float(subset_df["label"].mean()),
    "label_min": float(subset_df["label"].min()),
    "label_max": float(subset_df["label"].max()),
})
print(subset_df[["sentence1", "sentence2", "label", "label_bin"]].head(10))


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()

print({
    "loaded_model": model_name,
    "hidden_size": int(model.config.hidden_size),
    "device": device,
})


In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    masked = last_hidden_state * mask
    summed = masked.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

def encode_texts_manual(texts, batch_size=64, max_length=128):
    all_embeddings = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch_texts = texts[start:start + batch_size]
            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            encoded = {k: v.to(device) for k, v in encoded.items()}
            outputs = model(**encoded)
            pooled = mean_pool(outputs.last_hidden_state, encoded["attention_mask"])
            normalized = F.normalize(pooled, p=2, dim=1)
            all_embeddings.append(normalized.detach().cpu())
    return torch.cat(all_embeddings, dim=0).numpy()

sentences1 = subset_df["sentence1"].tolist()
sentences2 = subset_df["sentence2"].tolist()
labels = subset_df["label"].to_numpy(dtype=np.float32)

emb1 = encode_texts_manual(sentences1, batch_size=batch_size, max_length=max_length)
emb2 = encode_texts_manual(sentences2, batch_size=batch_size, max_length=max_length)

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)

print({
    "emb1_shape": emb1.shape,
    "emb2_shape": emb2.shape,
    "cosine_min": float(cosine_similarity.min()),
    "cosine_max": float(cosine_similarity.max()),
    "pred_min": float(predicted_score_0_5.min()),
    "pred_max": float(predicted_score_0_5.max()),
})


In [ ]:
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic

results_df = subset_df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["absolute_error"] = np.abs(results_df["predicted_score_0_5"] - results_df["label"])
results_df["signed_error"] = results_df["predicted_score_0_5"] - results_df["label"]

per_bin_error_df = results_df.groupby("label_bin", sort=True).agg(
    count=("label", "size"),
    label_mean=("label", "mean"),
    pred_mean=("predicted_score_0_5", "mean"),
    mae=("absolute_error", "mean"),
    rmse=("signed_error", lambda s: float(np.sqrt(np.mean(np.square(s))))),
    bias=("signed_error", "mean"),
).reset_index()

lowest_pred_df = results_df.sort_values(["predicted_score_0_5", "absolute_error", "label"], ascending=[True, True, True]).head(5).reset_index(drop=True)
highest_pred_df = results_df.sort_values(["predicted_score_0_5", "absolute_error", "label"], ascending=[False, True, False]).head(5).reset_index(drop=True)
lowest_label_df = results_df.sort_values(["label", "absolute_error", "predicted_score_0_5"], ascending=[True, True, True]).head(5).reset_index(drop=True)
highest_label_df = results_df.sort_values(["label", "absolute_error", "predicted_score_0_5"], ascending=[False, True, False]).head(5).reset_index(drop=True)

print({
    "pearson_correlation": float(pearson_corr),
    "spearman_correlation": float(spearman_corr),
    "mae": float(results_df["absolute_error"].mean()),
})
print(per_bin_error_df)


In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"subset_num_examples: {len(subset_df)}")
print(f"subset_rule: balanced_random_sample_{target_subset_size}_examples_evenly_across_{num_bins}_label_bins_with_seed_{seed}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"mean_absolute_error: {results_df['absolute_error'].mean():.6f}")
print(f"max_absolute_error: {results_df['absolute_error'].max():.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")

print("per_label_bin_error:")
print(per_bin_error_df.to_dict(orient="records"))

print("lowest_pred_examples:")
print(lowest_pred_df[["sentence1", "sentence2", "label", "predicted_score_0_5", "absolute_error", "label_bin"]].to_dict(orient="records"))

print("highest_pred_examples:")
print(highest_pred_df[["sentence1", "sentence2", "label", "predicted_score_0_5", "absolute_error", "label_bin"]].to_dict(orient="records"))

print("lowest_label_examples:")
print(lowest_label_df[["sentence1", "sentence2", "label", "predicted_score_0_5", "absolute_error", "label_bin"]].to_dict(orient="records"))

print("highest_label_examples:")
print(highest_label_df[["sentence1", "sentence2", "label", "predicted_score_0_5", "absolute_error", "label_bin"]].to_dict(orient="records"))
